# Google ColabでのSAM3データ生成とYOLO学習

このノートブックでは、SAM3を使用してYOLO用のデータセットを生成し、YOLOv8モデルの学習を行います。
オリジナルリポジトリ: https://github.com/Guch1120/sam3_for_YOLO_learning

## 0. 環境セットアップ
リポジトリのクローンと必要なライブラリのインストールを行います。

### 0.1 Gitクローンとディレクトリ設定

In [ ]:
# リポジトリをクローン
!git clone https://github.com/Guch1120/sam3_for_YOLO_learning.git

# インポート競合を避けるため、パッケージルート（リポジトリ内部）へ移動
%cd sam3_for_YOLO_learning/sam3

### 0.2 システム依存関係

In [ ]:
# OpenCVなどで必要なシステムライブラリをインストール
!apt-get update && apt-get install -y libgl1-mesa-glx

### 0.3 Pythonライブラリ (Core)

In [ ]:
# Pythonライブラリのインストール
# pyproject.tomlの[dependencies]と[project.optional-dependencies] (notebooks) を含めてインストール
# bashでの解釈エラーを防ぐためバージョン指定を引用符で囲っています
!pip install ultralytics "timm>=1.0.17" "numpy<2.0" tqdm "ftfy==6.1.1" regex "iopath>=0.1.10" typing_extensions huggingface_hub pycocotools decord opencv-python einops scikit-image scikit-learn ipywidgets matplotlib

### 0.4 SAM3のインストール

In [ ]:
import sys
import os

# --- 再起動後のディレクトリ確認と自動修正 ---
# Colab再起動後はカレントディレクトリが /content に戻るため、正しい場所へ移動します
TARGET_DIR = "/content/sam3_for_YOLO_learning/sam3"
if os.getcwd() != TARGET_DIR:
    if os.path.exists(TARGET_DIR):
        os.chdir(TARGET_DIR)
        print(f"(自動修正) ディレクトリを {TARGET_DIR} に変更しました")
    else:
        print(f"警告: ターゲットディレクトリ {TARGET_DIR} が見つかりません。セル0.1 (Git Clone) を再実行してください。")

try:
    import sam3
    # 依存関係(decord)のチェックも含める
    import decord
    
    if not hasattr(sam3, 'build_sam3_image_model'):
        print("警告: インポートされたsam3が不完全のようです。再起動しても治らない場合は再インストールが必要です。")
        raise ImportError("Incomplete sam3 package")
    print("sam3は既にインストールされておりインポート可能です。再起動ループを避けるためインストールをスキップします。")
except ImportError as e:
    print(f"インストール中... (理由: {e})")
    # 正しいディレクトリにいるので、カレントディレクトリ (.) からインストール
    if os.path.exists("pyproject.toml"):
        !pip install -e .
        print("インストール完了。もしnumpyの更新によりランタイム再起動の警告が出た場合は、再起動してください。")
        print("出力の 'Restart Session' ボタン、またはメニューの 'ランタイム > セッションを再起動' をクリックしてください。")
        print("再起動後、このセルを再度実行してください。自動でディレクトリ修正とスキップ判定が行われます。")
    else:
        print("エラー: pyproject.toml が見つかりません。インストールできません。セル0.1を実行してディレクトリを作成してください。")

### 0.5 Hugging Face ログイン (SAM3利用に必須)
SAM3は「Gated Model（利用制限付きモデル）」です。ダウンロードするには認証が必要です。

**Step 1: アクセス権の取得**
1. [https://huggingface.co/facebook/sam3](https://huggingface.co/facebook/sam3) にアクセスし、ライセンス条項に **Agree (同意)** してください。
2. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) で **New Token** を作成します (Type: Read)。作成されたトークン（`hf_...`で始まる文字列）をコピーしてください。

**Step 2: Colabにトークンを設定 (推奨)**
1. Colab左サイドバーの **鍵アイコン (シークレット)** をクリックします。
2. 新しいシークレットを追加します:
    *   名前: `HF_TOKEN`
    *   値: (コピーしたトークンを貼り付け)
3. このシークレットの「ノートブックからのアクセス」を有効にします（スイッチをON）。

**Step 3: 以下のセルを実行**
シークレットから自動的にトークンを読み込みます。設定されていない場合は手動入力を求められます。

In [ ]:
from huggingface_hub import login
from google.colab import userdata

print("Hugging Faceの認証を確認中...")
try:
    # Colab Secrets (Name: HF_TOKEN) からトークン取得を試みる
    token = userdata.get('HF_TOKEN')
    if token:
        print("Colab Secretsに HF_TOKEN が見つかりました。ログインします...")
        login(token)
    else:
        raise ValueError("No secret found")
except Exception as e:
    print("HF_TOKEN シークレットが設定されていないか、アクセスできません。対話ログインに切り替えます。")
    print("プロンプトが表示されたら Hugging Face Access Token を入力してください。")
    from huggingface_hub import notebook_login
    notebook_login()

## 1. 設定 (Configuration)
パスやパラメータを設定します。

In [ ]:
import os
from google.colab import drive

# --- パス設定 ---
# Google Driveを使用する場合は True に設定してください
# ローカル（Colab内、またはリポジトリ内）を使用する場合は False に設定してください
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    print("Using Google Drive...")
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive"
    
    # --- Driveのパス ---
    INPUT_IMAGE_ROOT = os.path.join(DRIVE_ROOT, "sam3_data/images_root")
    OUTPUT_DIR = os.path.join(DRIVE_ROOT, "sam3_data/result")
else:
    print("Using Local Colab Storage...")
    # --- ローカルColabのパス ---
    # リポジトリのルートは /content/sam3_for_YOLO_learning です
    REPO_ROOT = "/content/sam3_for_YOLO_learning"
    
    # 入力: 元画像が保存されている場所
    # リクエストに基づき、sam3/sam3_for_YOLO_Learning フォルダ内を指すように調整
    INPUT_IMAGE_ROOT = os.path.join(REPO_ROOT, "sam3/sam3_for_YOLO_Learning/result/images")
    
    # 出力: 結果の保存先
    OUTPUT_DIR = os.path.join(REPO_ROOT, "sam3/sam3_for_YOLO_Learning/result/output")
    
    os.makedirs(INPUT_IMAGE_ROOT, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SAM3 設定 ---
TEXT_PROMPTS = [
    "long table",
    "square table with long legs",
    "black box",
    "shelf"
]
CONFIDENCE_THRESHOLD = 0.5

# --- YOLO 学習設定 ---
YOLO_MODEL_NAME = "yolov8m.pt"
YOLO_EPOCHS = 50
YOLO_BATCH_SIZE = 16
YOLO_IMAGE_SIZE = 640
YOLO_WORKERS = 2

print("設定完了")
print(f"モード: {'Google Drive' if USE_GOOGLE_DRIVE else 'Local Colab'}")
print(f"入力ルート: {INPUT_IMAGE_ROOT}")
print(f"出力ディレクトリ: {OUTPUT_DIR}")

## 2. アノテーション実行 (SAM3)
このブロックを実行して、SAM3を用いてラベルを生成します。

In [ ]:
import torch
import sam3
import yaml
import numpy as np
import random
import glob
from tqdm import tqdm
from PIL import Image, ImageDraw
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# --- ユーティリティ関数 ---
def convert_box_to_yolo(box, img_w, img_h):
    x1, y1, x2, y2 = box
    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    return xc, yc, w, h

def get_class_colors(class_names):
    random.seed(42)
    colors = {}
    for name in class_names:
        colors[name] = (
            random.randint(0, 255),
            random.randint(0, 255),
            random.randint(0, 255)
        )
    return colors

# --- アノテーションロジック ---
def run_annotation():
    # 1. デバイス設定
    if torch.cuda.is_available():
        device = "cuda"
        print("デバイス: CUDAを使用します")
    else:
        device = "cpu"
        print("デバイス: CPUを使用します")

    # 2. 出力ディレクトリの準備
    IMAGES_TRAIN_DIR = os.path.join(OUTPUT_DIR, "images", "train")
    LABELS_TRAIN_DIR = os.path.join(OUTPUT_DIR, "labels", "train")
    IMAGES_VAL_DIR = os.path.join(OUTPUT_DIR, "images", "val")
    LABELS_VAL_DIR = os.path.join(OUTPUT_DIR, "labels", "val")
    VIS_OUT_DIR    = os.path.join(OUTPUT_DIR, "visualize")
    DATA_YAML_PATH = os.path.join(OUTPUT_DIR, "data.yaml")

    for d in [IMAGES_TRAIN_DIR, LABELS_TRAIN_DIR, IMAGES_VAL_DIR, LABELS_VAL_DIR, VIS_OUT_DIR]:
        os.makedirs(d, exist_ok=True)

    # 3. モデルのロード
    print("SAM3モデルをロード中...")
    # Note: pip install -e . での構成を前提としています
    try:
        # インストールされたパッケージ内のアセットを探す
        import sam3
        sam3_pkg_root = os.path.dirname(os.path.abspath(sam3.__file__))
        # 構成に応じて調整; クローンまたはダウンロードが必要な場合もある
        bpe_path = os.path.join(sam3_pkg_root, "..", "assets", "bpe_simple_vocab_16e6.txt.gz")
        if not os.path.exists(bpe_path):
             # リポジトリルートから実行している場合のフォールバック
             bpe_path = "./assets/bpe_simple_vocab_16e6.txt.gz"

        model = build_sam3_image_model(bpe_path=bpe_path, device=device)
        processor = Sam3Processor(model=model, confidence_threshold=CONFIDENCE_THRESHOLD, device=device)
    except Exception as e:
        print(f"モデルのロード中にエラーが発生しました: {e}")
        try:
             # 認証エラーの可能性がある場合のフォールバック
             print("認証状態を確認しています...")
             from huggingface_hub import login
             print("401エラーの場合は、セル0.5で正しくログインしているか確認してください。")
        except ImportError:
             pass
        return

    class_map = {name: idx for idx, name in enumerate(TEXT_PROMPTS)}
    class_colors = get_class_colors(TEXT_PROMPTS)

    # 4. 画像の収集 (再帰的)
    print(f"{INPUT_IMAGE_ROOT} から画像を検索中...")
    image_files = []
    for ext in ["jpg", "png", "jpeg", "JPG", "PNG", "JPEG"]:
        image_files.extend(glob.glob(os.path.join(INPUT_IMAGE_ROOT, "**", f"*.{ext}"), recursive=True))
    
    image_files = sorted(list(set(image_files)))
    if not image_files:
        print(f"{INPUT_IMAGE_ROOT} に画像が見つかりませんでした。パスを確認してください。")
        return

    # 5. 学習・検証データの分割
    random.seed(42)
    random.shuffle(image_files)
    split_idx = int(len(image_files) * 0.8)
    train_files = set(image_files[:split_idx])
    
    print(f"合計画像数: {len(image_files)}")
    print(f"学習用: {len(train_files)}, 検証用: {len(image_files) - len(train_files)}")

    # 6. 処理ループ
    for idx, image_path in enumerate(tqdm(image_files, desc="画像を処理中")):
        try:
            image_pil = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"{image_path} の読み込みに失敗しました: {e}")
            continue

        img_w, img_h = image_pil.size
        vis_image = image_pil.copy()
        draw = ImageDraw.Draw(vis_image)

        # 分割の判定
        is_train = image_path in train_files
        target_images_dir = IMAGES_TRAIN_DIR if is_train else IMAGES_VAL_DIR
        target_labels_dir = LABELS_TRAIN_DIR if is_train else LABELS_VAL_DIR

        # 画像の保存
        # フォルダ間のファイル名衝突を防ぐため、インデックスを含めた名前に変更
        out_image_name = f"sample_{idx:04d}.jpg"
        image_pil.save(os.path.join(target_images_dir, out_image_name))

        label_lines = []
        try:
            inference_state = processor.set_image(image_pil)
            for prompt in TEXT_PROMPTS:
                results = processor.set_text_prompt(prompt, inference_state)
                if len(results["boxes"]) == 0:
                    continue

                class_id = class_map[prompt]
                color = class_colors[prompt]

                for box in results["boxes"]:
                    xc, yc, w, h = convert_box_to_yolo(box, img_w, img_h)
                    label_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

                    # 可視化
                    x1, y1, x2, y2 = map(int, box)
                    draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
                    draw.text((x1, max(y1 - 10, 0)), prompt, fill=color)
        except Exception as e:
            print(f"{image_path} の推論に失敗しました: {e}")
            continue

        # ラベルの保存
        label_path = os.path.join(target_labels_dir, out_image_name.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            for line in label_lines:
                f.write(line + "\n")
        
        # 可視化画像の保存 (確認用として一箇所に保存)
        vis_image.save(os.path.join(VIS_OUT_DIR, out_image_name))

    # 7. data.yaml の作成
    data_yaml = {
        "path": os.path.abspath(OUTPUT_DIR),
        "train": "images/train",
        "val": "images/val",
        "names": TEXT_PROMPTS
    }
    with open(DATA_YAML_PATH, "w") as f:
        yaml.dump(data_yaml, f, allow_unicode=True)

    print("=== アノテーション完了 ===")
    print(f"データセット保存先: {OUTPUT_DIR}")

if __name__ == '__main__':
    run_annotation()

## 3. 学習実行 (YOLO)
生成したデータセットを用いてYOLOv8モデルを学習します。

In [ ]:
from ultralytics import YOLO

def run_training():
    data_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
    if not os.path.exists(data_yaml_path):
        print(f"エラー: {data_yaml_path} に data.yaml が見つかりません。先にアノテーションブロックを実行してください。")
        return

    # モデルのダウンロードまたはロード
    # Colabのローカルコンテンツにダウンロードされ、Driveの遅延を回避します
    print(f"YOLOモデルをロード中: {YOLO_MODEL_NAME}...")
    model = YOLO(YOLO_MODEL_NAME)

    print("学習を開始します...")
    results = model.train(
        data=data_yaml_path,
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMAGE_SIZE,
        batch=YOLO_BATCH_SIZE,
        workers=YOLO_WORKERS,
        project=os.path.join(OUTPUT_DIR, "runs"), # Google Drive (または指定のOUTPUT_DIR) に直接保存
        name="train_sam3_yolo",
        exist_ok=True,
        plots=True
    )
    print("学習完了。")
    print(f"結果は以下に保存されました: {os.path.join(OUTPUT_DIR, 'runs')}")

if __name__ == '__main__':
    run_training()